In [ ]:
# Check Pytorch installation
import torch, torchvision
print(torch.__version__, torch.cuda.is_available())

# Check MMDetection installation
import mmdet
print(mmdet.__version__)

# Check mmcv installation
from mmcv.ops import get_compiling_cuda_version, get_compiler_version
print(get_compiling_cuda_version())
print(get_compiler_version())


from pathlib import Path
from mmdet.apis import init_detector, inference_detector
from mmdet.registry import VISUALIZERS
import matplotlib.pyplot as plt
import tifffile
import mmcv
def find_weight_file(folder):
    pthfiles = [x for x in Path(folder).iterdir() if x.name.endswith('pth')]
    best = [x for x in pthfiles if 'best' in x.as_posix()]
    assert len(best)>0, 'No best weight found.'
    return best[0].as_posix()

# PARAMETERS:
FOLDER = '/Data_large/marine/PythonProjects/MMDET/checkpoints/Venus_b9/norm_test_vfnet_r18/20240610_151655_LR_0.001_BATCH_8_IMG_2304'
CFG_FILE = f'{FOLDER}/vfnet_r18.py'
PTH_FILE = find_weight_file(FOLDER)
# Choose to use a config and initialize the detector
config = CFG_FILE
# Setup a checkpoint file to load
checkpoint = PTH_FILE
# initialize the detector
model = init_detector(config, checkpoint, device='cuda:0')

visualizer = VISUALIZERS.build(model.cfg.visualizer)

visualizer.dataset_meta = model.dataset_meta

img_path = '/Data_large/marine/Datasets/VENuS/ds_L0/coarse_band_9/ASH_L0_15736_20200722_CoReg_mask_OK.tif'
try:
    img = tifffile.imread(img_path)
    if img is None:
        raise ValueError(f"Failed to read the image from {img_path}. Please check the file path.")
except Exception as e:
    raise ValueError(f"Failed to read the image from {img_path}. Error: {e}")

# Perform inference
result = inference_detector(model, img)

# img = mmcv.imconvert(img, 'bgr', 'rgb')
visualizer.add_datasample(
name='result',
image=img,
data_sample=result,
draw_gt=False,
pred_score_thr=0.3,
show=False)

img = visualizer.get_image()
plt.figure(dpi=300, figsize=(100,100))
plt.imshow(img)
plt.savefig('/Data_large/marine/PythonProjects/MMDET/results/figura.png')
plt.show()